# Experiment 15 — Low-data stress test: PatternSearchCV (eager) vs BayesHalvingSearchCV, and stratified vs random with a different seed (2026-07-19)

Two comparisons in one run, zones `[0.15%, 0.20%, 2%, 100%]` — one rung below Experiment 14's 0.2% floor, `verbose=2` on every arm (full per-decision debug log saved in this notebook's cell outputs).

1. **PatternSearchCV (eager, stratified) vs BayesHalvingSearchCV (stratified)** at this low-data regime — does eager's extra mesh contraction help or hurt at this fraction, and how does it compare to BHS's GP-EI search under the same aggressive ladder?
2. **PatternSearchCV (eager, stratified) vs PatternSearchCV (eager, random, `random_state=42`)** — Experiment 14 found `random` matched `stratified` at 0.2% with seed 0, contradicting the store-coverage theory. This repeats the test with a *different* seed (42, chosen arbitrarily - any seed != 0 serves the purpose) at an even more aggressive 0.15% starting fraction, to see whether seed 0 was simply lucky and a different seed finally shows `random` breaking down where the coverage side-analysis predicted it should.

All three arms: `n_starts=1`, official grid, `TimeSeriesSplit(5)`, MAE. Per-tier fit counts (how many evaluations landed at each rung of the ladder) reported for all three, computed directly from `cv_results_` (not from the verbose log, which is for inspection only).

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_12416\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 13.8 s


In [2]:
# --- common setup: grid, CV, zones ladder [0.15%, 0.20%, 2%, 100%] ---
import numpy as np
from collections import Counter
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from pattern_search_cv import BayesHalvingSearchCV, PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]
ZONES = [0.0015, 0.002, 0.02, 1.0]

def make_clf():
    return ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                               max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}
tscv = TimeSeriesSplit(n_splits=5)


def _summarize(label, search, wall):
    res = search.cv_results_
    n_res = np.asarray(res["n_resources"])
    tiers = Counter(int(v) for v in n_res)
    fit_work_total = float(np.sum(np.asarray(res["mean_fit_time"]) * 5))
    out = {
        "label": label, "wall": wall,
        "n_fits": len(res["params"]),
        "tiers": tiers,
        "equiv": float(np.sum(n_res / len(y))),
        "best": search.best_params_, "best_mae": -search.best_score_,
        "fit_work_total": fit_work_total,
        "zones_used": sorted(set(int(v) for v in n_res)),
    }
    print(f"\n{label:32s}: {out['n_fits']} evals, {out['equiv']:.3f} equiv, "
         f"{wall:.1f} s wall, best {out['best']} MAE {out['best_mae']:.3f}")
    for n_rows, count in sorted(tiers.items()):
        print(f"    {count:3d} fits @ {n_rows:>7d} rows ({n_rows / len(y):.4%})")
    return out


def run_psc(label, contraction, subsample, random_state):
    search = PatternSearchCV(make_clf(), param_grid,
                             scoring="neg_mean_absolute_error", cv=tscv,
                             n_jobs=-1, poll="opportunistic",
                             contraction=contraction, data_zones=ZONES,
                             subsample=subsample,
                             random_state=random_state, verbose=2)
    t0 = time.time()
    search.fit(X, y)
    wall = time.time() - t0
    return _summarize(label, search, wall)


def run_bhs(label, subsample, random_state, n_iter=25, promote_k=3):
    search = BayesHalvingSearchCV(make_clf(), param_grid,
                                  scoring="neg_mean_absolute_error", cv=tscv,
                                  n_jobs=-1, data_zones=ZONES,
                                  subsample=subsample, random_state=random_state,
                                  verbose=2, n_iter=n_iter, promote_k=promote_k)
    t0 = time.time()
    search.fit(X, y)
    wall = time.time() - t0
    out = _summarize(label, search, wall)
    out["n_cache_hits"] = search.n_cache_hits_
    return out

In [3]:
PSC_eager_strat = run_psc("PSC eager / stratified", "eager", "stratified", 0)

[pattern_search_cv] PatternSearchCV: optimizing metric = neg_mean_absolute_error


[pattern_search_cv] cv = TimeSeriesSplit


[pattern_search_cv]   max_features : [2, 3, 4]


[pattern_search_cv]   n_estimators : [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260]


[pattern_search_cv]   max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


...

[pattern_search_cv] fit_time per fold: [ 22.28825569  41.42079902  63.22180104  91.96261144 119.60877633]


[pattern_search_cv] fit_time: 67.700449


[pattern_search_cv] score_time per fold: [3.54394579 3.06162453 3.65100932 3.5877254  3.45023084]


[pattern_search_cv] score_time: 3.458907



PSC eager / stratified          : 30 evals, 10.030 equiv, 2080.1 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038
     20 fits @     628 rows (0.1501%)
     10 fits @  418416 rows (100.0000%)


In [4]:
BHS_strat = run_bhs("BayesHalvingSearchCV / stratified", "stratified", 0)

[pattern_search_cv] BayesHalvingSearchCV: optimizing metric = neg_mean_absolute_error


[pattern_search_cv] cv = TimeSeriesSplit


[pattern_search_cv]   max_features : [2, 3, 4]


[pattern_search_cv]   n_estimators : [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260]


[pattern_search_cv]   max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


...

[pattern_search_cv] fit_time per fold: [ 19.92978573  39.59246778  61.23765492  86.11896729 105.87625599]


[pattern_search_cv] fit_time: 62.551026


[pattern_search_cv] score_time per fold: [3.20854998 3.17343879 3.06755781 3.34400129 3.08368254]


[pattern_search_cv] score_time: 3.175446



BayesHalvingSearchCV / stratified: 28 evals, 3.038 equiv, 1022.1 s wall, best {'max_features': 4, 'n_estimators': 140, 'max_depth': 15} MAE 866.197
     25 fits @     628 rows (0.1501%)
      3 fits @  418416 rows (100.0000%)


In [5]:
PSC_eager_rand42 = run_psc("PSC eager / random (seed=42)", "eager", "random", 42)

[pattern_search_cv] PatternSearchCV: optimizing metric = neg_mean_absolute_error


[pattern_search_cv] cv = TimeSeriesSplit


[pattern_search_cv]   max_features : [2, 3, 4]


[pattern_search_cv]   n_estimators : [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260]


[pattern_search_cv]   max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


...

[pattern_search_cv] fit_time per fold: [ 28.86435342  57.38088179  87.1440804  122.11320472 175.5621922 ]


[pattern_search_cv] fit_time: 94.212943


[pattern_search_cv] score_time per fold: [4.28051281 4.25799251 4.48902392 4.40265203 4.73499751]


[pattern_search_cv] score_time: 4.433036



PSC eager / random (seed=42)    : 31 evals, 6.171 equiv, 1939.7 s wall, best {'max_features': 4, 'n_estimators': 170, 'max_depth': 17} MAE 807.968
     11 fits @     628 rows (0.1501%)
      7 fits @     837 rows (0.2000%)
      7 fits @    8369 rows (2.0002%)
      6 fits @  418416 rows (100.0000%)


In [6]:
# --- standard comparison table, fits-per-tier rows added ---
arms = [PSC_eager_strat, BHS_strat, PSC_eager_rand42]
all_tiers = sorted(set().union(*(a['tiers'].keys() for a in arms)))

print("=" * 100)
print(f"zones ladder {ZONES}")
cols = [a['label'] for a in arms]
print(f"{'':24s}" + "".join(f"{c:>26s}" for c in cols))
print(f"{'total evaluations':24s}" + "".join(f"{a['n_fits']:>26d}" for a in arms))
for n_rows in all_tiers:
    frac = n_rows / len(y)
    row_label = f'fits @ {frac:.4%}'
    print(f"{row_label:24s}" + "".join(f"{a['tiers'].get(n_rows, 0):>26d}" for a in arms))
print(f"{'full-fit equivalents':24s}" + "".join(f"{a['equiv']:>26.3f}" for a in arms))
print(f"{'wall-clock (s)':24s}" + "".join(f"{a['wall']:>26.1f}" for a in arms))
print(f"{'summed fit work (s)':24s}" + "".join(f"{a['fit_work_total']:>26.1f}" for a in arms))
print(f"{'best point':24s}" + "".join(
    f"{str((a['best']['max_features'], a['best']['n_estimators'], a['best']['max_depth'])):>26s}"
    for a in arms))
print(f"{'best CV MAE':24s}" + "".join(f"{a['best_mae']:>26.3f}" for a in arms))
print(f"{'zones used (rows)':24s}" + "".join(f"{str(a['zones_used']):>26s}" for a in arms))
print()
print(f"BHS cache hits: {BHS_strat['n_cache_hits']}")

zones ladder [0.0015, 0.002, 0.02, 1.0]
                            PSC eager / stratifiedBayesHalvingSearchCV / stratifiedPSC eager / random (seed=42)
total evaluations                               30                        28                        31
fits @ 0.1501%                                  20                        25                        11
fits @ 0.2000%                                   0                         0                         7
fits @ 2.0002%                                   0                         0                         7
fits @ 100.0000%                                10                         3                         6
full-fit equivalents                        10.030                     3.038                     6.171
wall-clock (s)                              2080.1                    1022.1                    1939.7
summed fit work (s)                         4686.8                    1572.4                    3790.7
best point              

## Summary

Fill in after running:

1. **PSC eager vs BHS, low-data stratified**: which reaches a good MAE at fewer full-fit equivalents at this more aggressive 0.15% starting fraction? Does either one fail to converge to a reasonable optimum?
2. **PSC eager stratified vs random (seed=42)**: does `random` break down with this seed where seed 0 (Experiment 14) didn't? Or does it hold up again, further weakening the coverage-based prediction?
3. **Who breaks first?** Any arm that fails to reach a competitive MAE, or needs a disproportionate number of full-data fits to converge, at this more aggressive ladder.